# SAGE-Like Local Recommendation Loop

This notebook demonstrates the public adaptive-calibration interface without requiring a local SAGE installation, PBS, or cluster access. Users provide parameter ranges, a target profile, and optional metrics files. If no initial metrics are available, the session returns a cold-start Latin-hypercube design for the requested number of initial runs.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

from autocalibration import AdaptiveCalibrationSession, ParameterSpec


## 1. Define Parameters And Target

These names mirror a small SAGE-like calibration surface, but the loop is generic.

In [ ]:
parameters = [
    ParameterSpec('SfrEfficiency', 0.1, 1.0),
    ParameterSpec('FeedbackReheatingEpsilon', 0.5, 5.0),
]
target_profile = ROOT / 'examples' / 'sage_like_target_profile.json'


## 2. Cold-Start LHS When Metrics Are Not Available

If the user has not run any simulations yet, request the number of initial runs and pass those recommended parameter vectors to the simulator manually.

In [ ]:
cold_start_session = AdaptiveCalibrationSession.from_files(
    parameters=parameters,
    target_profile=target_profile,
)
initial_runs = cold_start_session.recommend(n=4, seed=1)
initial_runs


## 3. Start From A Metrics File

After simulations finish, collect parameter columns and metric columns in a CSV file. The example below reads `examples/sage_like_initial_metrics.csv`.

In [ ]:
session = AdaptiveCalibrationSession.from_files(
    parameters=parameters,
    target_profile=target_profile,
    metrics_file=ROOT / 'examples' / 'sage_like_initial_metrics.csv',
    metric_columns=['stellar_mass_density', 'gas_fraction'],
)
session.losses()


In [ ]:
recommendations = session.recommend(n=2, seed=5)
recommendations


## 4. Continue After New Simulation Results

Append new metrics from another CSV/JSON/HDF5 file, then request the next recommendations. HDF5 loading is optional and requires `pip install h5py`.

In [ ]:
session.add_observations_from_file(
    ROOT / 'examples' / 'sage_like_new_metrics_iter001.csv',
    metric_columns=['stellar_mass_density', 'gas_fraction'],
)
next_recommendations = session.recommend(n=2, seed=6)
next_recommendations
